# CodeReviewAgent — интеллектуальный помощник по ревью кода

Этот проект демонстрирует, как построить интеллектуального помощника по ревью кода на фреймворке HelloAgents.

## 📖 Инструкция

- **Быстрый старт**: запустите быструю демонстрацию в «Части 0»
- **Полный функционал**: последовательно выполните части 1–7 для полного цикла ревью кода


---

## Часть 0: Быстрая демонстрация ⚡

Если хотите быстро познакомиться с проектом, запустите эту упрощённую версию.


In [ ]:
# Быстрая демонстрация — импорт и конфигурация
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import ast
import os

# Параметры LLM
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "your_api_key_here"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

print("✅ Библиотеки импортированы, конфигурация завершена")


[output cleared — rerun cell after translation]


In [14]:
# Быстрая демонстрация — простой инструмент анализа кода
class QuickAnalysisTool(Tool):
    def __init__(self):
        super().__init__(
            name="quick_analysis",
            description="Быстрый анализ структуры Python-кода"
        )
    
    def run(self, parameters: Dict[str, Any]) -> str:
        code = parameters.get("code", "")
        if not code:
            return "Ошибка: код не может быть пустым"
        
        try:
            tree = ast.parse(code)
            functions = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
            classes = [n.name for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
            return f"Найдено {len(classes)} классов и {len(functions)} функций: {', '.join(functions)}"
        except Exception as e:
            return f"Ошибка разбора кода: {str(e)}"
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="Python-код для анализа",
                required=True
            )
        ]

print("✅ Инструмент определён")


[output cleared — rerun cell after translation]


In [15]:
# Быстрая демонстрация — реестр инструментов и агент
from hello_agents import ToolRegistry

quick_registry = ToolRegistry()
quick_registry.register_tool(QuickAnalysisTool())

quick_agent = SimpleAgent(
    name="Быстрый помощник по ревью",
    llm=HelloAgentsLLM(),
    system_prompt="Вы помощник по ревью кода: анализируйте код инструментами и давайте краткие рекомендации.",
    tool_registry=quick_registry
)

print("✅ Агент создан")
print(f"✅ Доступные инструменты: {list(quick_registry._tools.keys())}")


[output cleared — rerun cell after translation]


In [16]:
test_code = """
def hello():
    print("Hello")

def world():
    print("World")

class Greeter:
    def greet(self):
        hello()
        world()
"""

print("=== Быстрая демонстрация: анализ тестового кода ===")
result = quick_agent.run(f"Проанализируй этот код:\n{test_code}")
print(result)
print("\n✅ Быстрая демонстрация завершена!")
print("\n💡 Подсказка: продолжите выполнение ячеек ниже для полного функционала")


[output cleared — rerun cell after translation]


---

# Полная система ревью кода

Ниже — полная система ревью с расширенными возможностями анализа.


## Часть 1: Настройка окружения


In [ ]:
# Импорт необходимых библиотек
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import ast
import os

# Параметры LLM
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "your_api_key_here"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

print("✅ Окружение настроено")
print(f"✅ Модель: {os.getenv('LLM_MODEL_ID')}")
print(f"✅ API-адрес: {os.getenv('LLM_BASE_URL')}")


[output cleared — rerun cell after translation]


## Часть 2: Определение инструментов анализа кода


In [18]:
class CodeAnalysisTool(Tool):
    """Инструмент статического анализа кода"""

    def __init__(self):
        super().__init__(
            name="code_analysis",
            description="Анализ структуры, сложности и потенциальных проблем Python-кода"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """Анализ кода и возврат результата"""
        code = parameters.get("code", "")
        if not code:
            return "Ошибка: код не может быть пустым"
        
        try:
            tree = ast.parse(code)

            functions = [node for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
            classes = [node for node in ast.walk(tree) if isinstance(node, ast.ClassDef)]

            result = {
                "число_функций": len(functions),
                "число_классов": len(classes),
                "число_строк": len(code.split('\n')),
                "список_функций": [f.name for f in functions],
                "список_классов": [c.name for c in classes]
            }

            return str(result)
        except SyntaxError as e:
            return f"Синтаксическая ошибка: {str(e)}"
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="Python-код для анализа",
                required=True
            )
        ]

print("✅ CodeAnalysisTool определён")


[output cleared — rerun cell after translation]


In [19]:
class StyleCheckTool(Tool):
    """Инструмент проверки стиля кода"""

    def __init__(self):
        super().__init__(
            name="style_check",
            description="Проверка соответствия кода PEP 8"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """Проверка стиля кода"""
        code = parameters.get("code", "")
        if not code:
            return "Ошибка: код не может быть пустым"
        
        issues = []

        lines = code.split('\n')
        for i, line in enumerate(lines, 1):
            if len(line) > 79:
                issues.append(f"Строка {i}: более 79 символов")

            if line.startswith(' ') and not line.startswith('    '):
                if len(line) - len(line.lstrip()) not in [0, 4, 8, 12]:
                    issues.append(f"Строка {i}: некорректный отступ")

        if not issues:
            return "Стиль кода хороший, соответствует PEP 8"
        return "Обнаружены проблемы:\n" + "\n".join(issues)
    
    def get_parameters(self) -> List[ToolParameter]:
        return [
            ToolParameter(
                name="code",
                type="string",
                description="Python-код для проверки",
                required=True
            )
        ]

print("✅ StyleCheckTool определён")


[output cleared — rerun cell after translation]


## Часть 3: Создание агента


In [20]:
from hello_agents import ToolRegistry

tool_registry = ToolRegistry()
tool_registry.register_tool(CodeAnalysisTool())
tool_registry.register_tool(StyleCheckTool())

llm = HelloAgentsLLM()

system_prompt = """Вы — опытный эксперт по ревью кода. Ваши задачи:

1. Использовать инструмент code_analysis для анализа структуры кода
2. Использовать инструмент style_check для проверки стиля
3. На основе результатов предоставить подробный отчёт о ревью

Отчёт должен включать:
- анализ структуры кода
- проблемы стиля
- потенциальные баги
- рекомендации по оптимизации производительности
- рекомендации по лучшим практикам

Выводите отчёт в формате Markdown."""

agent = SimpleAgent(
    name="Помощник по ревью кода",
    llm=llm,
    system_prompt=system_prompt,
    tool_registry=tool_registry
)

print("✅ Агент создан")
print(f"Имя агента: {agent.name}")
print(f"Доступные инструменты: {list(tool_registry._tools.keys())}")


[output cleared — rerun cell after translation]


## Часть 4: Чтение примера кода


In [21]:
with open("data/sample_code.py", "r", encoding="utf-8") as f:
    sample_code = f.read()

print("=== Код для ревью ===")
print(sample_code)
print("\n" + "="*50 + "\n")


[output cleared — rerun cell after translation]


## Часть 5: Выполнение ревью кода


In [22]:
print("=== Начало ревью кода ===")
review_result = agent.run(f"Проведи ревью следующего Python-кода:\n\n```python\n{sample_code}\n```")

print(review_result)


[output cleared — rerun cell after translation]


## Часть 6: Сохранение отчёта о ревью


In [23]:
with open("outputs/review_report.md", "w", encoding="utf-8") as f:
    f.write(review_result)

print("\n✅ Отчёт о ревью сохранён в outputs/review_report.md")


[output cleared — rerun cell after translation]


## Часть 7: Итоги и перспективы

### Реализованные функции
- ✅ Анализ структуры кода
- ✅ Проверка стиля PEP 8
- ✅ Генерация интеллектуального отчёта о ревью

### Встреченные трудности
- Как точно разобрать структуру Python-кода
- Как спроектировать промпт для качественного отчёта от LLM

### Направления будущих улучшений
- Поддержка других языков программирования
- Обнаружение уязвимостей безопасности
- Интеграция дополнительных инструментов статического анализа
- Пакетное ревью нескольких файлов
